# Spendly: model-training trial

This Colab notebook uses the combined **synthetic** raw CSV to demonstrate the complete learning workflow: inspect, clean, explore, engineer features, train, evaluate, predict, and save models. The two exercises mirror the project: unusual-spending detection and weekly spending forecasting.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib tensorflow

In [ ]:
from pathlib import Path
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print('Libraries ready')

## 1. Load the raw combined CSV
Upload `combined_finance_raw.csv` when prompted. If it is already beside the notebook, the cell loads it directly.

In [ ]:
csv_path = Path('combined_finance_raw.csv')
if not csv_path.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path(next(iter(uploaded)))

raw = pd.read_csv(csv_path, low_memory=False)
print(f'Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
display(raw.head())

In [ ]:
quality = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_count': raw.isna().sum(),
    'missing_percent': (raw.isna().mean() * 100).round(1),
    'unique_values': raw.nunique(dropna=True),
}).sort_values('missing_percent', ascending=False)
display(quality)
print('Duplicate transaction IDs:', raw['transaction_id'].duplicated().sum())

## 2. Make a processed copy
The raw dataframe is never overwritten. We parse dates, coerce numbers, normalise categories, and turn blank anomaly labels into `False` only in `data`.

In [ ]:
data = raw.copy()
date_columns = ['transaction_timestamp', 'user_date_of_birth', 'budget_period_start', 'budget_period_end']
for column in date_columns:
    data[column] = pd.to_datetime(data[column], errors='coerce')

for column in ['amount', 'user_monthly_income', 'budget_amount']:
    data[column] = pd.to_numeric(data[column], errors='coerce')

data['category_clean'] = data['category'].astype(str).str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
data['merchant_missing'] = data['merchant'].isna().astype(int)
data['is_anomaly_target'] = data['is_anomaly'].fillna(False).astype(bool).astype(int)
data = data.drop_duplicates('transaction_id').dropna(subset=['transaction_timestamp', 'amount', 'user_id'])
data = data[data['amount'] > 0].sort_values('transaction_timestamp').reset_index(drop=True)
print(f'Processed shape: {data.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
category_spend = (data[data.transaction_type.eq('expense')].groupby('category')['amount'].sum().nlargest(10).sort_values())
category_spend.plot(kind='barh', ax=axes[0], color='#176b5b', title='Top expense categories')
monthly = data.assign(month=data.transaction_timestamp.dt.to_period('M').dt.to_timestamp()).groupby(['month', 'transaction_type'])['amount'].sum().unstack(fill_value=0)
monthly.plot(ax=axes[1], color=['#b42318', '#176b5b'], marker='o', title='Monthly cash flow')
axes[0].set_xlabel('KES'); axes[1].set_ylabel('KES')
plt.tight_layout(); plt.show()

## 3. Isolation Forest: unusual-spending detection
Isolation Forest is unsupervised: it learns what is uncommon without using the anomaly target during fitting. We use the controlled synthetic labels only for evaluation.

In [ ]:
anomaly_frame = data.copy()
anomaly_frame['log_amount'] = np.log1p(anomaly_frame['amount'])
anomaly_frame['hour'] = anomaly_frame['transaction_timestamp'].dt.hour
anomaly_frame['day_of_week'] = anomaly_frame['transaction_timestamp'].dt.dayofweek
anomaly_frame['is_expense'] = anomaly_frame['transaction_type'].eq('expense').astype(int)
anomaly_frame['is_recurring_num'] = anomaly_frame['is_recurring'].fillna(False).astype(int)
anomaly_features = ['log_amount', 'hour', 'day_of_week', 'is_expense', 'is_recurring_num', 'merchant_missing']
X_anomaly = anomaly_frame[anomaly_features].fillna(0)
known_rate = anomaly_frame['is_anomaly_target'].mean()
contamination = float(np.clip(known_rate, 0.005, 0.15))
anomaly_model = IsolationForest(n_estimators=250, contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1)
anomaly_frame['iforest_raw'] = anomaly_model.fit_predict(X_anomaly)
anomaly_frame['predicted_anomaly'] = anomaly_frame['iforest_raw'].eq(-1).astype(int)
anomaly_frame['anomaly_score'] = -anomaly_model.decision_function(X_anomaly)
print('Contamination used:', round(contamination, 4))
print(classification_report(anomaly_frame['is_anomaly_target'], anomaly_frame['predicted_anomaly'], digits=3))

In [ ]:
display(anomaly_frame.nlargest(12, 'anomaly_score')[['transaction_timestamp', 'user_id', 'amount', 'category', 'anomaly_score', 'is_anomaly_target', 'predicted_anomaly']])

## 4. Weekly spending forecast
For a clear classroom example, choose the user with the most expense records, aggregate weekly spending, build eight lag features, and keep the newest 20% strictly for testing. This avoids leaking future weeks into training.

In [ ]:
expenses = data[data.transaction_type.eq('expense')].copy()
trial_user = expenses['user_id'].value_counts().index[0]
weekly = (expenses[expenses.user_id.eq(trial_user)].set_index('transaction_timestamp')['amount'].resample('W-MON').sum().rename('weekly_spending').to_frame())
for lag in range(1, 9):
    weekly[f'lag_{lag}'] = weekly['weekly_spending'].shift(lag)
weekly['rolling_mean_4'] = weekly['weekly_spending'].shift(1).rolling(4).mean()
weekly['rolling_std_4'] = weekly['weekly_spending'].shift(1).rolling(4).std()
weekly_model = weekly.dropna().copy()
features = [f'lag_{lag}' for lag in range(1, 9)] + ['rolling_mean_4', 'rolling_std_4']
split = max(1, int(len(weekly_model) * 0.8))
train, test = weekly_model.iloc[:split], weekly_model.iloc[split:]
X_train, y_train = train[features], train['weekly_spending']
X_test, y_test = test[features], test['weekly_spending']
print('Trial user:', trial_user, '| train weeks:', len(train), '| test weeks:', len(test))

In [ ]:
forecast_models = {
    'Linear regression': LinearRegression(),
    'Random forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
}
rows, predictions = [], {}
for name, model in forecast_models.items():
    model.fit(X_train, y_train)
    predicted = np.maximum(0, model.predict(X_test))
    predictions[name] = predicted
    rows.append({
        'model': name,
        'MAE_KES': mean_absolute_error(y_test, predicted),
        'RMSE_KES': mean_squared_error(y_test, predicted) ** 0.5,
        'R2': r2_score(y_test, predicted) if len(y_test) > 1 else np.nan,
    })
results = pd.DataFrame(rows).sort_values('MAE_KES')
display(results.style.format({'MAE_KES':'{:,.0f}', 'RMSE_KES':'{:,.0f}', 'R2':'{:.3f}'}))

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(y_test.index, y_test.values, marker='o', linewidth=3, label='Actual', color='#17332d')
for name, predicted in predictions.items():
    plt.plot(y_test.index, predicted, marker='o', label=name)
plt.title(f'Weekly spending forecast — {trial_user}')
plt.ylabel('KES'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Optional LSTM
Run this section to compare a small sequence model. With a short history, a simple model can perform better; that is an important result, not a failure.

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_STATE)
values = weekly['weekly_spending'].values.astype('float32').reshape(-1, 1)
scaler = StandardScaler()
scaled = scaler.fit_transform(values)
window = 8
X_seq, y_seq = [], []
for index in range(window, len(scaled)):
    X_seq.append(scaled[index-window:index])
    y_seq.append(scaled[index])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)
seq_split = max(1, int(len(X_seq) * 0.8))
lstm = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(window, 1)),
    tf.keras.layers.LSTM(24),
    tf.keras.layers.Dense(1),
])
lstm.compile(optimizer='adam', loss='mse')
history = lstm.fit(X_seq[:seq_split], y_seq[:seq_split], validation_split=0.2, epochs=30, batch_size=8, verbose=0)
lstm_scaled = lstm.predict(X_seq[seq_split:], verbose=0)
lstm_pred = np.maximum(0, scaler.inverse_transform(lstm_scaled).ravel())
lstm_actual = scaler.inverse_transform(y_seq[seq_split:]).ravel()
print('LSTM MAE (KES):', round(mean_absolute_error(lstm_actual, lstm_pred), 2))
pd.DataFrame(history.history).plot(title='LSTM training history'); plt.show()

## 6. Next-week prediction and saved artifacts

In [ ]:
best_name = results.iloc[0]['model']
best_model = forecast_models[best_name]
next_week_prediction = max(0, float(best_model.predict(weekly_model[features].tail(1))[0]))
print(f'Best tabular model: {best_name}')
print(f'Illustrative next-week spending prediction: KES {next_week_prediction:,.2f}')
joblib.dump(best_model, 'trial_weekly_forecast.joblib')
joblib.dump(anomaly_model, 'trial_isolation_forest.joblib')
lstm.save('trial_lstm.keras')
print('Saved trial_weekly_forecast.joblib, trial_isolation_forest.joblib, and trial_lstm.keras')

## What to explain in your university demonstration

1. **Raw vs processed:** missing values were retained in the CSV and handled in a copy.
2. **No leakage:** forecasting used a chronological split and lagged values only.
3. **Baseline matters:** linear regression and random forest provide a fair comparison to LSTM.
4. **Anomaly ≠ fraud:** Isolation Forest only flags unusual patterns for review.
5. **Metrics:** MAE/RMSE are in KES; classification precision/recall describe anomaly detection.
6. **Synthetic limitation:** results demonstrate the pipeline and do not prove performance on real users.